# Data extraction notebook (futures + climate/disaster)

In [ ]:

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data/raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "1960-01-01"
END_DATE = "2026-01-01"



## Futures monthly data

In [ ]:

tickers = {"corn": "ZC=F", "soybean": "ZS=F", "wheat": "ZW=F"}
frames = []

for commodity, ticker in tickers.items():
    daily = yf.download(ticker, start=START_DATE, end=END_DATE, auto_adjust=False, progress=False)
    if daily.empty:
        print(f"No data for {commodity}")
        continue
    if isinstance(daily.columns, pd.MultiIndex):
        daily.columns = daily.columns.get_level_values(0)
    price_col = "Adj Close" if "Adj Close" in daily.columns else "Close"

    monthly = daily[[price_col]].resample("ME").last().copy()
    monthly["futures_return"] = np.log(monthly[price_col]).diff()
    monthly["commodity"] = commodity
    monthly = monthly.reset_index().rename(columns={"Date": "date", price_col: "futures_price"})
    frames.append(monthly)

futures_df = pd.concat(frames, ignore_index=True)
futures_df.to_csv(RAW_DIR / "futures_monthly.csv", index=False)
print(futures_df.head())


## ENSO + FEMA disaster features

In [ ]:

oni_url = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"
season_to_month = {
    "DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4,
    "AMJ": 5, "MJJ": 6, "JJA": 7, "JAS": 8,
    "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12,
}

oni = pd.read_csv(oni_url, sep=r"\s+")
oni = oni.rename(columns={"SEAS": "season", "YR": "year", "ANOM": "enso_oni"})
oni["month"] = oni["season"].map(season_to_month)
oni["date"] = pd.to_datetime(dict(year=oni["year"], month=oni["month"], day=1)) + pd.offsets.MonthEnd(0)
climate_disaster_df = oni[["date", "enso_oni"]].copy()

def fetch_fema(start_year=2010, chunk_size=5000):
    base = "https://www.fema.gov/openfema-data-hub-api/disaster-declarations-summaries-v2"
    rows, skip = [], 0
    while True:
        params = {
            "$select": "declarationDate,incidentType",
            "$filter": f"declarationDate ge '{start_year}-01-01T00:00:00.000Z'",
            "$top": chunk_size,
            "$skip": skip,
            "$orderby": "declarationDate asc",
        }
        r = requests.get(base, params=params, timeout=60)
        r.raise_for_status()
        batch = r.json().get("DisasterDeclarationsSummaries", [])
        if not batch:
            break
        rows.extend(batch)
        if len(batch) < chunk_size:
            break
        skip += chunk_size
    return pd.DataFrame(rows)

fema = fetch_fema(2010)
if not fema.empty:
    fema["date"] = pd.to_datetime(fema["declarationDate"], errors="coerce").dt.to_period("M").dt.to_timestamp("M")
    fema["incidentType"] = fema["incidentType"].fillna("Unknown")

    monthly_total = fema.groupby("date").size().rename("disaster_event_count").reset_index()
    storm = (
        fema.assign(flag=fema["incidentType"].str.lower().eq("severe storm(s)"))
        .groupby("date")["flag"].sum().rename("disaster_storm_count").reset_index()
    )
    flood = (
        fema.assign(flag=fema["incidentType"].str.lower().eq("flood"))
        .groupby("date")["flag"].sum().rename("disaster_flood_count").reset_index()
    )
    fire = (
        fema.assign(flag=fema["incidentType"].str.lower().eq("fire"))
        .groupby("date")["flag"].sum().rename("disaster_fire_count").reset_index()
    )

    disaster_monthly = monthly_total.merge(storm, on="date", how="left").merge(flood, on="date", how="left").merge(fire, on="date", how="left")
    climate_disaster_df = climate_disaster_df.merge(disaster_monthly, on="date", how="left")

climate_disaster_df = climate_disaster_df.sort_values("date")
climate_disaster_df.to_csv(RAW_DIR / "climate_disaster_monthly.csv", index=False)
climate_disaster_df.tail()
